In [1]:
import pandas as pd

In [3]:
dataset = pd.read_json("../datasets/events_dataset_v2.json")

In [4]:
dataset["era"].unique()

array(['حقبة النشأة', 'حقبة الخلافة الأموية', 'حقبة ملوك الطوائف',
       'حقبة الدول المغاربية', 'حقبة مملكة غرناطة'], dtype=object)

In [5]:
import pandas as pd
import json

# Load your events data (assuming you have the JSON from previous responses)
with open('../datasets/events_dataset_v2.json', 'r', encoding='utf-8') as f:
    events = json.load(f)

# Create a DataFrame
df = pd.DataFrame(events)

# Display class distribution
print(df['era'].value_counts())

# Example structure:
# - text: event description + event name (input)
# - label: era (target)
df['text'] = df['event_name'] + " - " + df['description']

era
حقبة الدول المغاربية    211
حقبة الخلافة الأموية    206
حقبة النشأة             203
حقبة ملوك الطوائف       198
حقبة مملكة غرناطة       174
Name: count, dtype: int64


In [6]:
# to keep only bahers that have more than 3000 samples
data = df
dfs = pd.DataFrame(data)
column_name = 'era'
min_count = 5
counts = dfs[column_name].value_counts()
frequent_values = counts[counts > min_count].index
filtered_df = dfs[dfs[column_name].isin(frequent_values)]

In [7]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
filtered_df['label'] = label_encoder.fit_transform(filtered_df['era'])

# Save label mapping for later use
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
# with open('../datasets/events_dataset1.json', 'w', encoding='utf-8') as f:
#     json.dump(label_mapping, f, ensure_ascii=True, indent=2)

In [8]:
filtered_df["era"].value_counts()

era
حقبة الدول المغاربية    211
حقبة الخلافة الأموية    206
حقبة النشأة             203
حقبة ملوك الطوائف       198
حقبة مملكة غرناطة       174
Name: count, dtype: int64

In [9]:
label_mapping

{'حقبة الخلافة الأموية': 0,
 'حقبة الدول المغاربية': 1,
 'حقبة النشأة': 2,
 'حقبة ملوك الطوائف': 3,
 'حقبة مملكة غرناطة': 4}

In [10]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    filtered_df['text'].tolist(), 
    filtered_df['label'].tolist(), 
    test_size=0.1, 
    random_state=42,
    stratify=filtered_df['label']
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, 
    temp_labels, 
    test_size=0.5, 
    random_state=42,
    stratify=temp_labels
)

print(f"Training: {len(train_texts)} samples")
print(f"Validation: {len(val_texts)} samples")
print(f"Testing: {len(test_texts)} samples")

Training: 892 samples
Validation: 50 samples
Testing: 50 samples


In [11]:
model_name = "../pre_trained_models/marbert"
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize texts
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# Create PyTorch Dataset
class EventDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EventDataset(train_encodings, train_labels)
val_dataset = EventDataset(val_encodings, val_labels)
test_dataset = EventDataset(test_encodings, test_labels)

/home/hussam/python_envs/complete_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Get number of unique eras
num_labels = len(label_encoder.classes_)

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels
)

# Define metrics for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': accuracy, 'f1': f1}

# Training arguments
training_args = TrainingArguments(
    output_dir='../models/events_era_classification_models/events_era_classification_train',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,  # Use if GPU supports
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 881.07it/s, Materializing param=bert.pooler.dense.weight]                                
BertForSequenceClassification LOAD REPORT from: ../pre_trained_models/marbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different tas

In [13]:
# Initialize trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# Train
# trainer.train()

# Save the best model
# trainer.save_model("./best_model_1000")
# tokenizer.save_pretrained("./best_model_1000")

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.797912,0.756093,0.780000,0.775028
2,0.729546,0.673383,0.780000,0.772089
3,0.788544,0.906223,0.580000,0.551062
4,0.729060,0.844326,0.680000,0.677165
5,0.669750,0.628629,0.760000,0.759252


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

TrainOutput(global_step=560, training_loss=0.7390487040792193, metrics={'train_runtime': 42.4313, 'train_samples_per_second': 105.111, 'train_steps_per_second': 13.198, 'total_flos': 144396358729560.0, 'train_loss': 0.7390487040792193, 'epoch': 5.0})

In [17]:
trainer.save_model("../models/events_era_classification_models/events_era_classification_train/best")
tokenizer.save_pretrained("../models/events_era_classification_models/events_era_classification_train/best")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


('../models/events_era_classification_models/events_era_classification_train/best/tokenizer_config.json',
 '../models/events_era_classification_models/events_era_classification_train/best/tokenizer.json')

In [16]:
# Evaluate on test set
test_results = trainer.predict(test_dataset)
print(f"Test Accuracy: {test_results.metrics['test_accuracy']:.4f}")
print(f"Test F1 Score: {test_results.metrics['test_f1']:.4f}")

# Detailed classification report
predictions = np.argmax(test_results.predictions, axis=1)
print("\nClassification Report:")
print(classification_report(
    test_labels, 
    predictions, 
    target_names=label_encoder.classes_
))

Test Accuracy: 0.5800
Test F1 Score: 0.5701

Classification Report:
                      precision    recall  f1-score   support

حقبة الخلافة الأموية       0.71      0.45      0.56        11
حقبة الدول المغاربية       1.00      0.80      0.89        10
         حقبة النشأة       0.42      0.80      0.55        10
   حقبة ملوك الطوائف       0.67      0.20      0.31        10
   حقبة مملكة غرناطة       0.46      0.67      0.55         9

            accuracy                           0.58        50
           macro avg       0.65      0.58      0.57        50
        weighted avg       0.66      0.58      0.57        50



In [17]:
# import json
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
# from sklearn.preprocessing import LabelEncoder
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
# import torch

# # ============================================
# # 1. LOAD YOUR DATA
# # ============================================

# # Load your events data (assuming you saved it as JSON)
# # with open('../datasets/events_dataset.json', 'r', encoding='utf-8') as f:
# #     events = json.load(f)
# events = filtered_df
# # Convert to DataFrame
# df = pd.DataFrame(events)

# # Create text column (combine event name and description)
# df['text'] = df['event_name'] + ' - ' + df['description']

# print(f"Total samples: {len(df)}")
# print(f"Eras: {df['era'].unique()}")
# print(df['era'].value_counts())

# # ============================================
# # 2. ENCODE LABELS
# # ============================================

# label_encoder = LabelEncoder()
# df['label'] = label_encoder.fit_transform(df['era'])

# class_names = label_encoder.classes_
# print(f"\nClasses: {class_names}")
# print(f"Number of classes: {len(class_names)}")

# # ============================================
# # 3. SPLIT DATA (SAME AS TRAINING)
# # ============================================

# from sklearn.model_selection import train_test_split

# # Split into train and temp (70-30)
# train_texts, temp_texts, train_labels, temp_labels = train_test_split(
#     df['text'].tolist(),
#     df['label'].tolist(),
#     test_size=0.3,
#     random_state=42,
#     stratify=df['label']
# )

# # Split temp into validation and test (50-50)
# val_texts, test_texts, val_labels, test_labels = train_test_split(
#     temp_texts,
#     temp_labels,
#     test_size=0.5,
#     random_state=42,
#     stratify=temp_labels
# )

# print(f"\nTrain: {len(train_texts)}")
# print(f"Validation: {len(val_texts)}")
# print(f"Test: {len(test_texts)}")

# # ============================================
# # 4. LOAD YOUR TRAINED MODEL
# # ============================================

# # Path to your saved model (change this to your actual path)
# model_path = "./best_model"  # or wherever you saved your model

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModelForSequenceClassification.from_pretrained(model_path)

# # Move model to appropriate device
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)
# model.eval()

# print(f"Model loaded from {model_path}")
# print(f"Using device: {device}")

# # ============================================
# # 5. CREATE TEST DATASET
# # ============================================

# from torch.utils.data import Dataset, DataLoader

# class EventDataset(Dataset):
#     def __init__(self, texts, labels, tokenizer, max_length=512):
#         self.texts = texts
#         self.labels = labels
#         self.tokenizer = tokenizer
#         self.max_length = max_length
    
#     def __len__(self):
#         return len(self.texts)
    
#     def __getitem__(self, idx):
#         text = self.texts[idx]
#         label = self.labels[idx]
        
#         encoding = self.tokenizer(
#             text,
#             truncation=True,
#             padding='max_length',
#             max_length=self.max_length,
#             return_tensors='pt'
#         )
        
#         return {
#             'input_ids': encoding['input_ids'].flatten(),
#             'attention_mask': encoding['attention_mask'].flatten(),
#             'labels': torch.tensor(label, dtype=torch.long)
#         }

# # Create test dataset
# test_dataset = EventDataset(test_texts, test_labels, tokenizer)

# # ============================================
# # 6. GET PREDICTIONS
# # ============================================

# def get_predictions(model, dataset, batch_size=16):
#     """Get predictions for the dataset"""
#     dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
#     all_predictions = []
#     all_labels = []
#     all_probs = []
    
#     with torch.no_grad():
#         for batch in dataloader:
#             # Move batch to device
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['labels'].to(device)
            
#             # Forward pass
#             outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
#             # Get predictions
#             logits = outputs.logits
#             probs = torch.nn.functional.softmax(logits, dim=-1)
#             predictions = torch.argmax(logits, dim=-1)
            
#             # Store
#             all_predictions.extend(predictions.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())
#             all_probs.extend(probs.cpu().numpy())
    
#     return np.array(all_predictions), np.array(all_labels), np.array(all_probs)

# # Get predictions for test set
# predictions, true_labels, probabilities = get_predictions(model, test_dataset)

# print(f"\nPredictions shape: {predictions.shape}")
# print(f"True labels shape: {true_labels.shape}")

# # ============================================
# # 7. NOW y_true EXISTS! - CREATE CONFUSION MATRIX
# # ============================================

# # Calculate accuracy
# accuracy = np.mean(predictions == true_labels)
# print(f"\nTest Accuracy: {accuracy:.4f}")

# # Basic confusion matrix
# cm = confusion_matrix(true_labels, predictions)

# # Plot basic confusion matrix
# plt.figure(figsize=(12, 10))
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
# disp.plot(cmap='Blues', xticks_rotation=45, ax=plt.gca())
# plt.title('Confusion Matrix - Era Classification', fontsize=16, fontweight='bold')
# plt.tight_layout()
# plt.savefig('basic_confusion_matrix.png', dpi=300, bbox_inches='tight')
# plt.show()

# # ============================================
# # 8. ENHANCED CONFUSION MATRIX WITH PERCENTAGES
# # ============================================

# # Normalize confusion matrix
# cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# # Create figure with two subplots
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# # Plot 1: Absolute numbers
# sns.heatmap(cm, annot=True, fmt='d', 
#             xticklabels=class_names, yticklabels=class_names,
#             cmap='Blues', ax=ax1, cbar_kws={'label': 'Count'})
# ax1.set_xlabel('Predicted Era', fontsize=12)
# ax1.set_ylabel('True Era', fontsize=12)
# ax1.set_title('Confusion Matrix - Absolute Counts', fontsize=14, fontweight='bold')
# ax1.tick_params(axis='x', rotation=45)

# # Plot 2: Normalized (percentages)
# sns.heatmap(cm_normalized, annot=True, fmt='.2%', 
#             xticklabels=class_names, yticklabels=class_names,
#             cmap='YlOrRd', ax=ax2, cbar_kws={'label': 'Percentage'})
# ax2.set_xlabel('Predicted Era', fontsize=12)
# ax2.set_ylabel('True Era', fontsize=12)
# ax2.set_title('Confusion Matrix - Normalized (%)', fontsize=14, fontweight='bold')
# ax2.tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.savefig('confusion_matrix_both.png', dpi=300, bbox_inches='tight')
# plt.show()

# # ============================================
# # 9. PRINT CLASSIFICATION REPORT
# # ============================================

# print("\n" + "="*80)
# print("CLASSIFICATION REPORT")
# print("="*80)
# report = classification_report(true_labels, predictions, target_names=class_names)
# print(report)

# # Save report to file
# with open('classification_report.txt', 'w', encoding='utf-8') as f:
#     f.write(report)

# # ============================================
# # 10. PER-CLASS METRICS DATAFRAME
# # ============================================

# from sklearn.metrics import precision_recall_fscore_support

# precision, recall, f1, support = precision_recall_fscore_support(true_labels, predictions)

# metrics_df = pd.DataFrame({
#     'Era': class_names,
#     'Precision': precision,
#     'Recall': recall,
#     'F1-Score': f1,
#     'Support': support,
#     'Correct': [cm[i, i] for i in range(len(class_names))],
#     'Total': cm.sum(axis=1),
#     'Accuracy': [cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0 for i in range(len(class_names))]
# })

# print("\n" + "="*80)
# print("PER-CLASS METRICS")
# print("="*80)
# print(metrics_df.to_string(index=False))

# # Save to CSV
# metrics_df.to_csv('per_class_metrics.csv', index=False, encoding='utf-8-sig')

# # ============================================
# # 11. IDENTIFY MISCLASSIFIED EXAMPLES
# # ============================================

# # Find misclassified indices
# misclassified_indices = np.where(predictions != true_labels)[0]

# print(f"\nTotal misclassified: {len(misclassified_indices)} out of {len(true_labels)}")

# # Show some examples
# print("\n" + "="*80)
# print("MISCLASSIFIED EXAMPLES")
# print("="*80)

# for i, idx in enumerate(misclassified_indices[:10]):  # Show first 10
#     true_era = class_names[true_labels[idx]]
#     pred_era = class_names[predictions[idx]]
#     confidence = probabilities[idx][predictions[idx]]
    
#     print(f"\n{i+1}. Example {idx}:")
#     print(f"   True: {true_era}")
#     print(f"   Pred: {pred_era} (confidence: {confidence:.2%})")
#     print(f"   Text: {test_texts[idx][:150]}...")

# # ============================================
# # 12. CONFUSION MATRIX WITH ARABIC TITLES
# # ============================================

# # Arabic era names (customize as needed)
# arabic_era_names = [
#     'عصر الإمارة الأموية',
#     'عصر الخلافة الأموية',
#     'عصر الفتنة',
#     'عصر ملوك الطوائف الأول',
#     'عصر المرابطين',
#     'عصر الموحدين',
#     'عصر ملوك الطوائف الثاني',
#     'عصر مملكة غرناطة',
#     'عصر ما بعد السقوط'
# ]

# # Use actual class names that match your data
# actual_arabic_names = class_names  # Use your actual era names

# # Plot with Arabic labels
# plt.figure(figsize=(14, 12))
# sns.heatmap(cm_normalized, annot=cm, fmt='d', 
#             xticklabels=actual_arabic_names, yticklabels=actual_arabic_names,
#             cmap='viridis', annot_kws={'size': 10})

# plt.xlabel('الفترة المتوقعة / Predicted Era', fontsize=14, fontweight='bold')
# plt.ylabel('الفترة الحقيقية / True Era', fontsize=14, fontweight='bold')
# plt.title('مصفوفة الارتباك - تصنيف الفترات التاريخية\nConfusion Matrix - Historical Era Classification', 
#           fontsize=16, fontweight='bold')

# plt.xticks(rotation=45, ha='right', fontsize=10)
# plt.yticks(fontsize=10)
# plt.tight_layout()
# plt.savefig('arabic_confusion_matrix.png', dpi=300, bbox_inches='tight')
# plt.show()

# # ============================================
# # 13. FUNCTION TO EVALUATE NEW TEXTS
# # ============================================

# def predict_era(text, event_name=None):
#     """Predict era for a new text"""
#     if event_name:
#         input_text = f"{event_name} - {text}"
#     else:
#         input_text = text
    
#     # Tokenize
#     inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=512).to(device)
    
#     # Predict
#     with torch.no_grad():
#         outputs = model(**inputs)
#         logits = outputs.logits
#         probs = torch.nn.functional.softmax(logits, dim=-1)
#         pred_id = torch.argmax(logits, dim=-1).item()
#         confidence = probs[0][pred_id].item()
    
#     predicted_era = class_names[pred_id]
    
#     # Get top 3 predictions
#     top_probs, top_indices = torch.topk(probs[0], 3)
#     top_predictions = []
#     for i in range(3):
#         top_predictions.append({
#             'era': class_names[top_indices[i].item()],
#             'probability': top_probs[i].item()
#         })
    
#     return {
#         'predicted_era': predicted_era,
#         'confidence': confidence,
#         'top_3': top_predictions
#     }

# # Test the function
# test_result = predict_era(
#     "سقوط غرناطة في أيدي الإسبان وخروج آخر ملوك المسلمين",
#     "سقوط غرناطة"
# )
# print(f"\nTest prediction: {test_result}")

# print("\n" + "="*80)
# print("ANALYSIS COMPLETE")
# print("="*80)
# print("Files saved:")
# print("  - basic_confusion_matrix.png")
# print("  - confusion_matrix_both.png")
# print("  - arabic_confusion_matrix.png")
# print("  - classification_report.txt")
# print("  - per_class_metrics.csv")